<a href="https://colab.research.google.com/github/daniel-ant/Calculo_paneles_bifaciales_flotantes/blob/main/PanelSolarBifacialFlotante.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Autor: Daniel Antonio Álvarez

Fecha inicio: 10/05/2026

Esta herramienta necesita de la instalación de la librería pvlib para funcionar.

El propósito de esta herramienta es el cálculo de los parámetros energéticos y económicos
de un panel solar bifacial flotante y su comparación con su equivalente en tierra.

Página de pvlib:
https://pvlib-python.readthedocs.io/en/stable/user_guide/index.html

Primero importamos bibliotecas e instalamos en este entorno pvlib

In [ ]:
pip install pvlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 41.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
import math
import pandas as pd
import pvlib
from pvlib.location import Location
from pvlib import irradiance

Constantes

In [ ]:
K = 1.3805e-23   # [J/K] Constante de Boltzmann
q  = 1.602e-19    # [C] Carga elemental del electrón
Eg = 1.1    #[eV, del silicio] Ancho de banda prohibida que depende del material

Parámetros de prueba y por defecto

In [ ]:
LATITUD = 40.6544           # en °N  (ejemplo: Ávila)
LONGITUD = -4.6961          # en °E
ALTITUD = 1000               # metros sobre el nivel del mar
TIMEZONE = "Europe/Madrid" #  Zona horaria de Ávila --> https://en.wikipedia.org/wiki/List_of_tz_database_time_zones
NOMBRE = "Ávila" #Solo para indicar en los resultados la localización

In [ ]:
MODULO = {
    "long_panel" : 1.961,    #Longitud del panel (verticalmente) [m]
    "ancho_panel" : 1.134,    #Ancho del panel [m]
    "P_stc_W" : 520,    # Potencia máxima a STC [W]
    "V_oc_V" : 40.78,    # Tensión en circuito abierto [V]
    "I_sc_A" : 16.01,   # Intensidad de cortocircuito [A]
    "Ns" : 108,     # Células en serie
    "Np" : 1,       # Células en paralelo
    "idealidad" : 1.3,     # Factor de idealidad (A) --> Asumido
    "T_stc_C" : 25.0,    # Temperatura de referencia STC [°C]
    "G_stc_W_m2" : 1000,    # Irradiancia de referencia STC [W/m²]
    "bifacialidad" : 0.80,    # Fracción de eficiencia de la cara trasera
    "Coef_T_Isc" : 0.00720,   #Coeficiente de temperatura de la corriente de cortocircuito de la celda Isc [Amperios/°C]
    #"Coef_T_Isc" : 0.045/100 * 16.01,  # A/°C ≈ 0.00720 A/°C
  }

# Calcular area_m2 después de que long_panel y ancho_panel estén definidos
MODULO["area_m2"] = MODULO["long_panel"] * MODULO["ancho_panel"]    # Área del panel [m²]

In [ ]:
SISTEMA = {
    "superficie_agua_m2" : 10_000,  # Superficie total de la masa de agua [m²]
    "ocupacion"          : 0.20,    # Porcentaje del agua que ocupará la instalación solar
    "tilt_grados"        : 21,      # Inclinación de los paneles [°]
    "azimuth_grados"     : 180,     # Azimuth: 180 = orientación Sur
    "albedo_agua"        : 0.065,    # Albedo del agua (reflexión)
      "albedo_tierra"      : 0.25,    # Albedo típico de tierra/vegetación (se puede ajustar)
      "tipo_instalacion"   : "agua",  # 'agua' o 'tierra' - para seleccionar albedo y modelo de T_celula
      "tipo_calculo_temp" : "faiman",    #Permite elegir cómo calcular la temperatura de la célula en tierra
    "perdidas_sistema"   : 0.14,    # Pérdidas por inversor, cableado, etc. (14%)
}

In [ ]:
ECONOMIA = {
    "coste_inversion_EUR_kW" : 425,   # Inversión inicial €/kWp --> 0,35-0,50 €/Wp gama media bifacial
    "coste_OM_EUR_kW_ano"    : 22,  # Costes por operación y mantenimiento €/kWp/año
    "tasa_descuento"         : 0.051,  # Tasa de descuento para la localización de la instalación
    "vida_util_anos"         : 30,    # Vida útil en años
    "degradacion_anual"      : 0.0042,  # 0.42 %/año. La planta se degrada con el tiempo y produce menos
}

Obtener los datos de pvlib

In [ ]:
def descarga_datos_pvlib():
  global tmy
  print(f"Descargando datos meteorológicos de PVGIS de {NOMBRE}...")

  tmy, _ = pvlib.iotools.get_pvgis_tmy(
      latitude=LATITUD,
      longitude=LONGITUD,
      map_variables=True,   # nombra columnas con el estándar pvlib
      timeout=30,   #Evitamos errores por carga infinita
  )
  tmy.index = tmy.index.tz_convert(TIMEZONE)
  tmy = tmy.dropna()

  print(f"Descargados {len(tmy)} registros horarios.")    #Comprobamos los registros descargados

Cálculos para la irradiancia bifacial

In [ ]:
def calculos_irradiancia():
  global G_front, G_rear, sol
  ubicacion = Location(LATITUD, LONGITUD, TIMEZONE, ALTITUD)
  sol = ubicacion.get_solarposition(tmy.index)    #Posición del sol en cada hora
  airmass = ubicacion.get_airmass(solar_position=sol)   #Masa de aire
  dni_extra = pvlib.irradiance.get_extra_radiation(tmy.index)   #Irradiancia normal directa extraterrestre por cada hora

  # Determinar el valor de albedo a usar basado en el tipo de instalación
  if SISTEMA["tipo_instalacion"] == "agua":
    albedo_actual = SISTEMA["albedo_agua"]
  elif SISTEMA["tipo_instalacion"] == "tierra":
    albedo_actual = SISTEMA["albedo_tierra"]
  else:
    raise ValueError("Tipo de instalación no reconocido. Use 'agua' o 'tierra'.")

  #Cara frontal
  #Hecho de forma automática para obtener la iiradiancia frontal en la localización

  poa = irradiance.get_total_irradiance(
      surface_tilt = SISTEMA["tilt_grados"],   #Inclinación paneles
      surface_azimuth = SISTEMA["azimuth_grados"],    #Orientación paneles
  #Ángulo del sol obtenido
      solar_zenith = sol["apparent_zenith"],
      solar_azimuth = sol["azimuth"],
  #Irradiancia (Directa Normal, Global Horizontal y Difusa Horizontal, respectivamente)
      dni = tmy["dni"],
      ghi = tmy["ghi"],
      dhi = tmy["dhi"],

      dni_extra = dni_extra,
      airmass = airmass["airmass_relative"],
      albedo = albedo_actual,
      model = "haydavies",    #Modelo que vamos a usar
  )
  G_front = poa["poa_global"].clip(lower=0).values    #Convertimos en array y eliminamos valores negativos

  # Cara trasera

  factor_vista_trasero = (1 + np.cos(np.radians(SISTEMA["tilt_grados"]))) / 2
  G_rear = (albedo_actual * tmy["ghi"] * factor_vista_trasero).clip(lower=0).values

Irradiancia final ambas caras

final = frontal + f. bifacialidad * trasera

In [ ]:
def irradiancia_final():
  global G_ef
  G_ef = G_front + MODULO["bifacialidad"] * G_rear

### Modelos de Temperatura del Panel

**1. Modelo de Agua:**
$$T_{cel} = 2.048 + 0.9458 \cdot T_{amb} + 0.0215 \cdot G_{ef} - 1.2376 \cdot V_{viento}$$

**2. Modelo de Tierra (Faiman - pvlib):**
El modelo de Faiman asume que la diferencia de temperatura sobre el ambiente es proporcional a la irradiancia e inversamente proporcional a la pérdida de calor por viento:

$$T_{cel} = T_{amb} + \frac{G_{ef}}{u_0 + u_1 \cdot V_{viento}}$$

Donde:
* $u_0$: Coeficiente de transferencia de calor independiente del viento ($25.0 \, W/m^2K$).
* $u_1$: Coeficiente de transferencia de calor dependiente del viento ($6.84 \, W/m^3sK$).

**3. Modelo de Tierra (Lineal - Simplificado):**
$$T_{cel} = T_{amb} + (45 - 20) \cdot \frac{G_{ef}}{800}$$

In [ ]:
def _temperatura_panel_agua(T_amb, G_ef, V_viento):
  return 2.048 + 0.9458 * T_amb + 0.0215 * G_ef - 1.2376 * V_viento

def temperatura_panel():
    global T_celula, T_lineal, T_faiman
    T_amb    = tmy["temp_air"].values   #Obtenemos temperatura del aire
    V_viento = tmy["wind_speed"].values   #Obtenemos velocidad del viento

    if SISTEMA["tipo_instalacion"] == "agua":
        T_celula = _temperatura_panel_agua(T_amb, G_ef, V_viento)

    elif SISTEMA["tipo_instalacion"] == "tierra":
        # Cálculo 1: Modelo Lineal (Inicial)
        T_lineal = T_amb + (45 - 20) * (G_ef / 800)

        # Cálculo 2: Modelo Faiman (pvlib). Los valores típicos por defecto que usamos son u0=25; u1=6.84
        T_faiman = pvlib.temperature.faiman(G_ef, T_amb, V_viento, u0=25.0, u1=6.84)

        # Aquí se decide el método por el que se calculará la temperatura en tierra
        if SISTEMA["tipo_calculo_temp"] == "faiman":
          T_celula = T_faiman
        else:
          T_celula = T_lineal

    else:
        raise ValueError("Tipo de instalación no reconocido.")

Parámetros del método del diodo simple

Obtener potencia de los paneles bifaciales

Devuelve la potencia en el MPP [W] para un panel dado G (Irradiancia) [W/m²] y T [°C].

𝑃=𝐼 ∙𝑉=(𝑁𝑃 ∙ 𝐼𝑃𝐻−𝑁𝑃 ∙ 𝐼𝑆∙ [exp(𝑞∙𝑉𝑁𝑠 ∙ 𝐾 ∙𝐴 ∙ 𝑇𝑖)−1])∙𝑉

Irs = Isc / exp(q•Voc / Ns•K•A•Ti) - 1

Vt = Ns*k*A*Ti/q


In [ ]:
def potencia_panel(G, T_C):
  #Con irradiancia baja el panel no produce energía
    if G <= 1.0:
        return 0.0
#Obtener los datos del conjunto "Módulo"

    Ns = MODULO["Ns"]
    Np = MODULO["Np"]
    A = MODULO["idealidad"]

#Convertimos Tª a ºK para los cálculos
    T_K = T_C + 273.15
    T_stc = MODULO["T_stc_C"] + 273.15

# Corriente fotogenerada (varía con G y T)
    Iph = (MODULO["I_sc_A"] + MODULO["Coef_T_Isc"] * (T_K - T_stc)) * (G / MODULO["G_stc_W_m2"])

# Corriente de saturación (calculada en condiciones STC)
    Irs = MODULO["I_sc_A"] / (np.exp(q * MODULO["V_oc_V"] / (Ns * K * A * T_stc)) - 1.0)    #Poner módulo en I_sc_A para evitar resultados negativos
    Is = Irs * (T_K / T_stc) ** 3 * np.exp((q * Eg / (A * K)) * ((1.0 / T_stc) - (1.0 / T_K)))

#Comprobación de que las intensidades son positivas
    if Iph <= 0 or Is <= 0:
        return 0.0

# tensión térmica del módulo, para simplificar el cálculo de la potencia
    Vt = Ns * A * K * T_K / q

# dP/dV = 0  →  Newton-Raphson sobre la función f(V) = dP/dV
    Voc_ap = Vt * np.log((Np * Iph) / (Np * Is) + 1.0)    #V circuito abierto
    V = 0.8 * Voc_ap    #0,8 ya que el voltaje a la máxima potencia suele ser entre el 70-80% del voltaje a circuito abierto. Asumimos 80%
    #Así hemos obtenido la potencia máxima de la función potencia respecto del voltaje


#   Usamos Newton-Rapson para encontrar el punto = 0 de dP/dV
    for _ in range(60):   #Solo lo ejecutamos 60 veces para evitar bucles infinitos y fallos si no encuentra solución
        exp_term = np.exp(V / Vt)   #Cálculo del exponencial
        I_v = Np * Iph - Np * Is * (exp_term - 1.0)   #Cálculo de la intensidad de salida del panel usando la línea anterior. I en el título del bloque
        dIdV = -Np * Is * exp_term / Vt   #Derivada de corriente respecto al voltaje. Buscamos I máxima
        f = I_v + V * dIdV    #P = V * I --> f es la derivada de lo anterior. Buscamos la "f" donde V = 0
        df = 2.0 * dIdV + V * (-Np * Is * exp_term / (Vt ** 2))   #Derivada de la función f de la línea anterior

        if abs(df) < 1e-14:   #Si df de la línea anterior es prácticamente 0 (para evitar errores al dividir por 0) fin del bucle for
            break
# np.clip(): Ajusta el valor de V_nuevo para que se mantenga dentro de un rango físico razonable (entre el 1% y el 99.5% del voltaje de circuito abierto aproximado Voc_ap). Esto ayuda a la estabilidad y convergencia del algoritmo.
        V_nuevo = float(np.clip(V - f / df, 0.01 * Voc_ap, 0.995 * Voc_ap))   #Cálculo del nuevo V que maximiza potencia
        if abs(V_nuevo - V) < 1e-7:   #Si V_nuevo es muy parecido al V inicial --> fin del bucle for y V = V_nuevo
            V = V_nuevo
            break
        V = V_nuevo   #Si los dos "if" anteriores no se ejecutan, V_nuevo = V

    I_mpp = max(Np * Iph - Np * Is * (np.exp(V / Vt) - 1.0), 0.0)   #Calculamos la máxima intensidad a partir del voltaje obtenido
    P = max(I_mpp * V, 0.0)   #Calculamos la potencia máxima
    return P    #Devolvemos la potencia

Simulación horaria hora a hora

In [ ]:
def simulacion_horaria():
  global E_hora_Wh, n_paneles, P_instalada_W, P_instalada_kW, area_paneles, area_ocupada_por_un_panel_con_sombra

  # Calcular la elevación solar mínima al mediodía solar a lo largo del año usando pvlib.
  daily_max_solar_elevation = sol.groupby(sol.index.date)['apparent_elevation'].max()
  # Se descartan valores negativos que daban problemas
  daily_max_solar_elevation = daily_max_solar_elevation[daily_max_solar_elevation > 0]
  min_solar_noon_elevation = daily_max_solar_elevation.min()

  # Calcular la distancia horizontal en el suelo requerida por panel para evitar el autosombreado,
  # considerando la inclinación y el ángulo de elevación solar mínimo.
  # Suma de la proyección horizontal del panel y la sombra proyectada por su altura vertical.
  long_sombra = (MODULO["long_panel"] * np.cos(np.radians(SISTEMA["tilt_grados"]))) + \
                ((MODULO["long_panel"] * np.sin(np.radians(SISTEMA["tilt_grados"]))) / np.tan(np.radians(min_solar_noon_elevation)))

  area_ocupada_por_un_panel_con_sombra = MODULO["ancho_panel"] * long_sombra
  n_paneles    = int((SISTEMA["superficie_agua_m2"] * SISTEMA["ocupacion"]) / area_ocupada_por_un_panel_con_sombra)

  # La variable 'area_paneles' ahora se usa para almacenar el área ocupada por un solo panel con sombra,
  # para distinguirla del área total de todos los paneles en la salida.
  area_paneles = area_ocupada_por_un_panel_con_sombra # Se mantiene el nombre de la variable para usos posteriores

  P_instalada_W  = n_paneles * MODULO["P_stc_W"]    #Nº paneles * Potencia nominal
  P_instalada_kW = P_instalada_W / 1000

  E_hora_Wh = np.zeros(len(tmy))    #Array en el que meteremos datos meteorológicos por hora con el tamaño que se considere adecuado (8760 huecos) lleno de ceros inicialmente

  for i in range(len(tmy)):   #Se repite 8760 veces, el nº de horas de un año
      P_panel = potencia_panel(G_ef[i], T_celula[i])    #Potencia en esa hora
      E_hora_Wh[i] = P_panel * n_paneles * (1 - SISTEMA["perdidas_sistema"])    #Energía por hora

Energía

𝐶𝐹=𝐸𝑎ñ𝑜Σ𝑃𝑆𝑇𝐶 ∙𝑁 ∙Δ𝑡𝑛𝑖=1 ∙100

In [ ]:
def calculos_energia():
  global E_año_Wh, E_año_kWh, E_año_MWh, E_año_GWh, CF, prod_esp
  E_año_Wh  = E_hora_Wh.sum()
  E_año_kWh = E_año_Wh / 1e3
  E_año_MWh = E_año_Wh / 1e6
  E_año_GWh = E_año_Wh / 1e9

  # Factor de Capacidad: energía real / energía si funcionara siempre al máximo
  CF = (E_año_kWh / (P_instalada_kW * 8760)) * 100    # [%]

  # Producción específica
  prod_esp = E_año_kWh / P_instalada_kW   # [kWh/kWp]

LCOE

𝐿𝐶𝑂𝐸=𝐼0+Σ𝑂𝑀(1+𝑑)𝑡𝑇𝑡=1Σ𝐸𝑎ñ𝑜(1+𝑑)𝑡

In [ ]:
def calculos_lcoe():
  global I0, OM, LCOE
  I0  = ECONOMIA["coste_inversion_EUR_kW"] * P_instalada_kW
  OM  = ECONOMIA["coste_OM_EUR_kW_ano"]    * P_instalada_kW
  d   = ECONOMIA["tasa_descuento"]
  T   = ECONOMIA["vida_util_anos"]
  dr = ECONOMIA["degradacion_anual"]

  anios = np.arange(1, T + 1)
  # Aseguramos que usamos E_año_MWh actualizado globalmente
  LCOE = (I0 + np.sum(OM / (1 + d)**anios)) / np.sum((E_año_MWh * (1 - dr)**anios) / (1 + d)**anios)
  #   Línea para saber cuándo acaban los cálculos al pulsar el botón de refrescar
  print("Cálculos completados, comprueba los resultados")

### Interfaz Gráfica para Entradas y Resultados

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Definición de Widgets de Entrada
style = {'description_width': 'initial'}

# Ubicación
w_lat = widgets.FloatText(value=LATITUD, description='Latitud (°N):', style=style)
w_lon = widgets.FloatText(value=LONGITUD, description='Longitud (°E):', style=style)
w_alt = widgets.FloatText(value=ALTITUD, description='Altitud (msnm):', style=style)
w_timezone = widgets.Text(value=TIMEZONE, description='Zona horaria:', style=style)
w_nom = widgets.Text(value=NOMBRE, description='Nombre Ciudad:', style=style)

# Módulo
w_long_panel = widgets.FloatText(value=MODULO['long_panel'], description='Longitud Panel (m):', style=style)
w_ancho_panel = widgets.FloatText(value=MODULO['ancho_panel'], description='Ancho Panel (m):', style=style)
w_p_stc = widgets.IntSlider(value=MODULO['P_stc_W'], min=10, max=1000, step=5, description='Potencia STC (W):', style=style)
w_area = widgets.FloatText(value=MODULO['area_m2'], description='Área Panel (m²):', style=style)
w_voc = widgets.FloatText(value=MODULO['V_oc_V'], description='V_oc (V):', style=style)
w_isc = widgets.FloatText(value=MODULO['I_sc_A'], description='I_sc (A):', style=style)
w_ns = widgets.IntText(value=MODULO['Ns'], description='Células Serie (Ns):', style=style)
w_np = widgets.IntText(value=MODULO['Np'], description='Células Paralelo (Np):', style=style)
w_ideal = widgets.FloatText(value=MODULO['idealidad'], description='Factor Idealidad:', style=style)
w_bifa = widgets.FloatSlider(value=MODULO['bifacialidad'], min=0, max=1.0, step=0.05, description='Bifacialidad:', style=style)
w_coef_isc = widgets.FloatText(value=MODULO['Coef_T_Isc'], description='Coef_T_Isc (A/°C):', style=style)

# Sistema
w_sup_agua = widgets.FloatText(value=SISTEMA['superficie_agua_m2'], description='Sup. Agua (m²):', style=style)
w_ocup = widgets.FloatSlider(value=SISTEMA['ocupacion'], min=0.01, max=1.0, step=0.01, description='Ocupación (frac):', style=style)
w_tilt = widgets.IntSlider(value=SISTEMA['tilt_grados'], min=0, max=90, description='Inclinación (°):', style=style)
w_azimuth = widgets.IntSlider(value=SISTEMA['azimuth_grados'], min=0, max=360, description='Azimuth (°):', style=style)
w_albedo = widgets.FloatSlider(value=SISTEMA['albedo_agua'], min=0, max=1.0, step=0.01, description='Albedo Agua:', style=style)
w_albedo_tierra = widgets.FloatSlider(value=SISTEMA['albedo_tierra'], min=0, max=1.0, step=0.01, description='Albedo tierra:', style=style)
w_perdidas = widgets.FloatSlider(value=SISTEMA['perdidas_sistema'], min=0, max=0.5, step=0.01, description='Pérdidas Sist.:', style=style)

w_tipo_instalacion = widgets.Dropdown(
    options=['agua', 'tierra'],
    value=SISTEMA['tipo_instalacion'],
    description='Tipo de Instalación:',
    style=style
)

w_tipo_calculo_temp = widgets.Dropdown(
    options=['faiman', 'lineal'],
    value=SISTEMA['tipo_calculo_temp'],
    description='Modelo Temp. Tierra:',
    style=style,
    layout=widgets.Layout(display='none' if SISTEMA['tipo_instalacion'] == 'agua' else 'flex')
)

# Lógica para mostrar/ocultar el selector de cálculo según el tipo de instalación
def on_tipo_change(change):
    if change['new'] == 'tierra':
        w_tipo_calculo_temp.layout.display = 'flex'
    else:
        w_tipo_calculo_temp.layout.display = 'none'

w_tipo_instalacion.observe(on_tipo_change, names='value')

# Economía
w_coste_inv = widgets.IntText(value=ECONOMIA['coste_inversion_EUR_kW'], description='Inversión (€/kWp):', style=style)
w_coste_om = widgets.FloatText(value=ECONOMIA['coste_OM_EUR_kW_ano'], description='O&M (€/kWp/año):', style=style)
w_tasa = widgets.FloatSlider(value=ECONOMIA['tasa_descuento'], min=0, max=0.2, step=0.01, description='Tasa Descuento:', style=style)
w_vida = widgets.IntText(value=ECONOMIA['vida_util_anos'], description='Vida Útil (años):', style=style)
w_degrad = widgets.FloatSlider(value=ECONOMIA['degradacion_anual'], min=0, max=0.05, step=0.0001, description='Degradación (frac):', style=style)

# Organización
box_ubicacion = widgets.VBox([w_nom, w_lat, w_lon, w_alt, w_timezone])
box_modulo = widgets.VBox([w_long_panel, w_ancho_panel, w_p_stc, w_area, w_voc, w_isc, w_ns, w_np, w_ideal, w_bifa, w_coef_isc])
box_sistema = widgets.VBox([w_sup_agua, w_ocup, w_tilt, w_azimuth, w_albedo, w_albedo_tierra, w_perdidas, w_tipo_instalacion, w_tipo_calculo_temp])
box_economia = widgets.VBox([w_coste_inv, w_coste_om, w_tasa, w_vida, w_degrad])

# Nombres de cada bloque
tab_inputs = widgets.Accordion(children=[box_ubicacion, box_modulo, box_sistema, box_economia])
tab_inputs.set_title(0, 'Ubicación')
tab_inputs.set_title(1, 'Módulo Fotovoltaico')
tab_inputs.set_title(2, 'Configuración del Sistema')
tab_inputs.set_title(3, 'Parámetros Económicos')

In [ ]:
# 2. Mostrar Formulario de Entrada
print("Ajusta los parámetros y pulsa el botón de refrescar resultados")
print("Se producirá error si no introduces una zona horaria correcta --> https://en.wikipedia.org/wiki/List_of_tz_database_time_zones")
display(tab_inputs)

Ajusta los parámetros y pulsa el botón de refrescar resultados
Se producirá error si no introduces una zona horaria correcta --> https://en.wikipedia.org/wiki/List_of_tz_database_time_zones


Accordion(children=(VBox(children=(Text(value='Ávila', description='Nombre Ciudad:', style=DescriptionStyle(de…

In [ ]:
# 3. Interfaz de Resultados
out_res = widgets.Output()

def actualizar_resultados(b):
    with out_res:
        clear_output()
        SEP = "=" * 65
        print(SEP)
        print("  RESULTADOS DEL PANEL SOLAR BIFACIAL FLOTANTE")
        print(SEP)
        print(f"\n  Localización     : {NOMBRE} ({LATITUD}°N, {LONGITUD}°E)")
        print(f"  Módulo           : ({MODULO['P_stc_W']} Wp)")

        print(f"\n  ── Sistema ──────────────────────────────────────")
        print(f"  Superficie agua  : {SISTEMA['superficie_agua_m2']:>10,.0f}  m²")
        print(f"  Área de paneles  : {area_ocupada_por_un_panel_con_sombra:>10,.0f}  m²  ({SISTEMA['ocupacion']*100:.1f}% de ocupación)")
        print(f"  Nº de paneles    : {n_paneles:>10,}")
        print(f"  Potencia instal. : {P_instalada_kW:>10,.1f}  kWp")
        print(f"  Inversión total  : {I0:>10,.0f}  €")

        print(f"\n  ── Producción energética ────────────────────────")
        print(f"  Energía anual    : {E_año_kWh:>10,.0f}  kWh/año")
        print(f"  Energía anual    : {E_año_MWh:>10,.1f}  MWh/año")
        print(f"  Energía anual    : {E_año_GWh:>10.4f}  GWh/año")
        print(f"  Prod. específica : {prod_esp:>10,.0f}  kWh/kWp")

        print(f"\n  ── Indicadores de rendimiento ───────────────────")
        print(f"  Factor Capacidad : {CF:>10.2f}  %")
        print(f"  LCOE             : {LCOE:>10.2f}  €/MWh")
  #Producción mensual
        print(f"\n{SEP}\n")

        # --- Resumen mensual añadido a la interfaz ---
        MESES = ["Ene","Feb","Mar","Abr","May","Jun",
                 "Jul","Ago","Sep","Oct","Nov","Dic"]

        df_temp = pd.DataFrame({
            "mes"       : tmy.index.month,
            "dia"       : tmy.index.dayofyear,
            "E_MWh"     : E_hora_Wh / 1e6,
            "G_front"   : G_front,
            "T_celula"  : T_celula,
        })

        print("  PRODUCCIÓN Y DATOS MEDIOS EN HORAS DE SOL")
        print(f"  {'Mes':>4} | {'E [MWh]':>9} | {'G_front sol':>11} | {'T_cél sol':>9} | {'T_máx med':>9}")
        print("  " + "-" * 58)
        for mes_idx, grupo in df_temp.groupby("mes"):
            # Se filtran solo las horas con sol para las medias de irradiancia y temperatura
            horas_sol = grupo[grupo['G_front'] > 0]

            g_media_sol = horas_sol['G_front'].mean() if not horas_sol.empty else 0.0
            t_media_sol = horas_sol['T_celula'].mean() if not horas_sol.empty else 0.0

            # Media de las máximas diarias (Tª máxima media)
            t_max_media = grupo.groupby("dia")['T_celula'].max().mean()

            print(f"  {MESES[mes_idx-1]:>4} | {grupo['E_MWh'].sum():>9.2f} | "
                  f"{g_media_sol:>11.1f} | {t_media_sol:>9.1f} | {t_max_media:>9.1f}")
        print(f"\n{SEP}")

btn_res = widgets.Button(description="Refrescar Resultados", button_style='success')
btn_res.on_click(actualizar_resultados)

display(btn_res, out_res)

Button(button_style='success', description='Refrescar Resultados', style=ButtonStyle())

Output()

In [ ]:
import pytz

def actualizar_resultados_completo(b):
    # 1. Vincular valores de los widgets a las variables globales
    vincular_datos_gui()

    # 2. Ejecutar la cadena de funciones de cálculo con manejo de excepciones
    try:
        descarga_datos_pvlib()
        calculos_irradiancia()
        irradiancia_final()
        temperatura_panel()
        simulacion_horaria()
        calculos_energia()
        calculos_lcoe()

        # 3. Mostrar los resultados finales en el output de la interfaz
        actualizar_resultados(b)
    except pytz.UnknownTimeZoneError:
        with out_res:
            clear_output()
            print("¡Error en el cálculo!")
            print("La zona horaria introducida no es válida.")
            print("Por favor, revisa la lista de zonas horarias en el siguiente enlace:")
            print("https://en.wikipedia.org/wiki/List_of_tz_database_time_zones")
    except Exception as e:
        with out_res:
            clear_output()
            print("¡Error en el cálculo!")
            print(f"Se produjo un error inesperado: {e}. Por favor, reinicie el programa")

# Asignamos la función al botón
btn_res.on_click(actualizar_resultados_completo, remove=False)

Aquí actualizamos los datos cada vez que se pulsa el botón, calculando todo de nuevo con los nuevos valores

In [ ]:
def vincular_datos_gui():
    """Actualiza las variables globales con los valores seleccionados en la GUI"""
    global LATITUD, LONGITUD, ALTITUD, TIMEZONE, NOMBRE
    LATITUD = w_lat.value
    LONGITUD = w_lon.value
    ALTITUD = w_alt.value
    TIMEZONE = w_timezone.value
    NOMBRE = w_nom.value

    MODULO['long_panel'] = w_long_panel.value
    MODULO['ancho_panel'] = w_ancho_panel.value
    MODULO['P_stc_W'] = w_p_stc.value
    MODULO['area_m2'] = MODULO['long_panel'] * MODULO['ancho_panel']
    MODULO['V_oc_V'] = w_voc.value
    MODULO['I_sc_A'] = w_isc.value
    MODULO['Ns'] = w_ns.value
    MODULO['Np'] = w_np.value
    MODULO['idealidad'] = w_ideal.value
    MODULO['bifacialidad'] = w_bifa.value
    MODULO['Coef_T_Isc'] = w_coef_isc.value

    SISTEMA['superficie_agua_m2'] = w_sup_agua.value
    SISTEMA['ocupacion'] = w_ocup.value
    SISTEMA['tilt_grados'] = w_tilt.value
    SISTEMA['azimuth_grados'] = w_azimuth.value
    SISTEMA['albedo_agua'] = w_albedo.value
    SISTEMA['albedo_tierra'] = w_albedo_tierra.value
    SISTEMA['perdidas_sistema'] = w_perdidas.value
    SISTEMA['tipo_instalacion'] = w_tipo_instalacion.value
    SISTEMA['tipo_calculo_temp'] = w_tipo_calculo_temp.value

    ECONOMIA['coste_inversion_EUR_kW'] = w_coste_inv.value
    ECONOMIA['coste_OM_EUR_kW_ano'] = w_coste_om.value
    ECONOMIA['tasa_descuento'] = w_tasa.value
    ECONOMIA['vida_util_anos'] = w_vida.value
    ECONOMIA['degradacion_anual'] = w_degrad.value

print("Función de vinculación. Se llamará automáticamente al refrescar resultados.")

Función de vinculación. Se llamará automáticamente al refrescar resultados.


Mostrar resultados con los valores predeterminados

In [ ]:
descarga_datos_pvlib()
calculos_irradiancia()
irradiancia_final()
temperatura_panel()
simulacion_horaria()
calculos_energia()
calculos_lcoe()

SEP = "=" * 65
print(SEP)
print("  RESULTADOS DEL PANEL SOLAR BIFACIAL FLOTANTE")
print(SEP)
print(f"\n  Localización     : {NOMBRE} ({LATITUD}°N, {LONGITUD}°E)")
print(f"  Módulo           : ({MODULO['P_stc_W']} Wp)")

print(f"\n  ── Sistema ──────────────────────────────────────")
print(f"  Superficie agua  : {SISTEMA['superficie_agua_m2']:>10,.0f}  m²")
print(f"  Área de paneles  : {area_paneles:>10,.0f}  m²  ({SISTEMA['ocupacion']*100:.1f}% de ocupación)")
print(f"  Nº de paneles    : {n_paneles:>10,}")
print(f"  Potencia instal. : {P_instalada_kW:>10,.1f}  kWp")
print(f"  Inversión total  : {I0:>10,.0f}  €")

print(f"\n  ── Producción energética ────────────────────────")
print(f"  Energía anual    : {E_año_kWh:>10,.0f}  kWh/año")
print(f"  Energía anual    : {E_año_MWh:>10,.1f}  MWh/año")
print(f"  Energía anual    : {E_año_GWh:>10.4f}  GWh/año")
print(f"  Prod. específica : {prod_esp:>10,.0f}  kWh/kWp")

print(f"\n  ── Indicadores de rendimiento ───────────────────")
print(f"  Factor Capacidad : {CF:>10.2f}  %")
print(f"  LCOE             : {LCOE:>10.2f}  €/MWh")
#Producción mensual
print(f"\n{SEP}\n")

# --- Resumen mensual añadido a la interfaz ---
MESES = ["Ene","Feb","Mar","Abr","May","Jun",
        "Jul","Ago","Sep","Oct","Nov","Dic"]

df_temp = pd.DataFrame({
  "mes"       : tmy.index.month,
  "dia"       : tmy.index.dayofyear,
  "E_MWh"     : E_hora_Wh / 1e6,
  "G_front"   : G_front,
  "T_celula"  : T_celula,
})

print("  PRODUCCIÓN Y DATOS MEDIOS EN HORAS DE SOL")
print(f"  {'Mes':>4} | {'E [MWh]':>9} | {'G_front sol':>11} | {'T_cél sol':>9} | {'T_máx med':>9}")
print("  " + "-" * 58)
for mes_idx, grupo in df_temp.groupby("mes"):
  # Filtramos solo las horas con sol para las medias de irradiancia y temperatura
  horas_sol = grupo[grupo['G_front'] > 0]

  g_media_sol = horas_sol['G_front'].mean() if not horas_sol.empty else 0.0
  t_media_sol = horas_sol['T_celula'].mean() if not horas_sol.empty else 0.0

  # Media de las máximas diarias (Tª máxima media)
  t_max_media = grupo.groupby("dia")['T_celula'].max().mean()

  print(f"  {MESES[mes_idx-1]:>4} | {grupo['E_MWh'].sum():>9.2f} | "
        f"{g_media_sol:>11.1f} | {t_media_sol:>9.1f} | {t_max_media:>9.1f}")
print(f"\n{SEP}")

Descargando datos meteorológicos de PVGIS de Ávila...
Descargados 8760 registros horarios.
Cálculos completados, comprueba los resultados
  RESULTADOS DEL PANEL SOLAR BIFACIAL FLOTANTE

  Localización     : Ávila (40.6544°N, -4.6961°E)
  Módulo           : (520 Wp)

  ── Sistema ──────────────────────────────────────
  Superficie agua  :     10,000  m²
  Área de paneles  :          4  m²  (20.0% de ocupación)
  Nº de paneles    :        537
  Potencia instal. :      279.2  kWp
  Inversión total  :    118,677  €

  ── Producción energética ────────────────────────
  Energía anual    :    383,238  kWh/año
  Energía anual    :      383.2  MWh/año
  Energía anual    :     0.3832  GWh/año
  Prod. específica :      1,372  kWh/kWp

  ── Indicadores de rendimiento ───────────────────
  Factor Capacidad :      15.67  %
  LCOE             :      38.25  €/MWh


  PRODUCCIÓN Y DATOS MEDIOS EN HORAS DE SOL
   Mes |   E [MWh] | G_front sol | T_cél sol | T_máx med
  ----------------------------------